# Notebook 12 — Baseline ML Models (Unified Corpus)

**Purpose** TF-IDF + Logistic Regression baselines on the unified EU+US corpus for two tasks: violation-type classification and severity-tier prediction. Includes cross-jurisdiction transfer experiments (train EU → test US and vice versa).

**Inputs**
- `data/unified_cases.csv` (2,754 rows — 337 US, 2,417 EU)

**Outputs**
- Classification reports for both tasks (in-distribution + cross-jurisdiction)

**Key decisions**
- Text feature: `decision_text` (FTC press release for US; GDPRhub facts+holding for EU)
- Violation type target: `coarse_label` (4 classes: Consent / Security / Transparency / Other)
- Severity target: `severity_tier` excluding No Fine (3 classes: Low / Medium / High)
- No Fine excluded from severity: non-monetary enforcement is qualitatively distinct from fined tiers
- Text source differences (press release vs. wiki summary) is a known confounder; transfer results reflect both legal-regime differences and domain shift. If we had real decision texts, this would still be relevant.

In [8]:
#imports
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack, csr_matrix
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("/Users/nic/Documents/MM2/data/unified_cases.csv")

print(f"Total rows: {len(df)}")
print(f"US: {(df['jurisdiction'] == 'US').sum()} | EU: {(df['jurisdiction'] != 'US').sum()}")
print(f"\nWith decision_text: {df['decision_text'].notna().sum()}")
print(f"With coarse_label:  {df['coarse_label'].notna().sum()}")
print(f"With severity_tier: {df['severity_tier'].notna().sum()}")

print("\n--- coarse_label distribution ---")
print(df['coarse_label'].value_counts())

print("\n--- severity_tier distribution ---")
print(df['severity_tier'].value_counts())

Total rows: 2754
US: 337 | EU: 2417

With decision_text: 2736
With coarse_label:  2627
With severity_tier: 2754

--- coarse_label distribution ---
coarse_label
Consent         943
Other           758
Security        587
Transparency    339
Name: count, dtype: int64

--- severity_tier distribution ---
severity_tier
No Fine    1379
Medium      845
High        404
Low         126
Name: count, dtype: int64


In [3]:
# Task 1: Violation Type (coarse_label)
task1 = df[df['decision_text'].notna() & df['coarse_label'].notna()].copy()
print(f"Task 1 corpus: {len(task1)} rows")
print(task1['coarse_label'].value_counts())

X = task1['decision_text'].astype(str)
y = task1['coarse_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=777, stratify=y
)

vec = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tf = vec.fit_transform(X_train)
X_test_tf  = vec.transform(X_test)

clf = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=777)
clf.fit(X_train_tf, y_train)

print("\n=== Task 1: Violation Type (in-distribution 80/20) ===")
print(classification_report(y_test, clf.predict(X_test_tf)))

Task 1 corpus: 2610 rows
coarse_label
Consent         941
Other           750
Security        581
Transparency    338
Name: count, dtype: int64

=== Task 1: Violation Type (in-distribution 80/20) ===
              precision    recall  f1-score   support

     Consent       0.71      0.76      0.73       188
       Other       0.70      0.67      0.68       150
    Security       0.80      0.78      0.79       116
Transparency       0.83      0.76      0.79        68

    accuracy                           0.74       522
   macro avg       0.76      0.74      0.75       522
weighted avg       0.74      0.74      0.74       522



In [4]:
eu = task1[task1['jurisdiction'] != 'US'].copy()
us = task1[task1['jurisdiction'] == 'US'].copy()

print(f"EU: {len(eu)} rows | US: {len(us)} rows")

# Train EU → Test US
vec_eu = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_eu_tf = vec_eu.fit_transform(eu['decision_text'].astype(str))
clf_eu = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
clf_eu.fit(X_eu_tf, eu['coarse_label'])

X_us_tf = vec_eu.transform(us['decision_text'].astype(str))
print("\n=== Task 1: Train EU → Test US ===")
print(classification_report(us['coarse_label'], clf_eu.predict(X_us_tf)))

# Train US → Test EU
vec_us = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_us_tf2 = vec_us.fit_transform(us['decision_text'].astype(str))
clf_us = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
clf_us.fit(X_us_tf2, us['coarse_label'])

X_eu_tf2 = vec_us.transform(eu['decision_text'].astype(str))
print("\n=== Task 1: Train US → Test EU ===")
print(classification_report(eu['coarse_label'], clf_us.predict(X_eu_tf2)))

EU: 2279 rows | US: 331 rows

=== Task 1: Train EU → Test US ===
              precision    recall  f1-score   support

     Consent       0.26      0.27      0.27        52
       Other       0.31      0.60      0.41        75
    Security       0.13      0.57      0.21        23
Transparency       0.09      0.02      0.03       181

    accuracy                           0.23       331
   macro avg       0.20      0.36      0.23       331
weighted avg       0.17      0.23      0.17       331


=== Task 1: Train US → Test EU ===
              precision    recall  f1-score   support

     Consent       0.43      0.00      0.01       889
       Other       0.32      0.03      0.06       675
    Security       0.00      0.00      0.00       558
Transparency       0.07      0.98      0.13       157

    accuracy                           0.08      2279
   macro avg       0.20      0.25      0.05      2279
weighted avg       0.27      0.08      0.03      2279



In [6]:
# Task 2: Severity Tier (Low / Medium / High — excluding No Fine)
task2 = df[
    df['decision_text'].notna() &
    df['severity_tier'].notna() &
    (df['severity_tier'] != 'No Fine')
].copy()

print(f"Task 2 corpus: {len(task2)} rows")
print(task2['severity_tier'].value_counts())

X2 = task2['decision_text'].astype(str)
y2 = task2['severity_tier']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=777, stratify=y2
)

vec2 = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X2_train_tf = vec2.fit_transform(X2_train)
X2_test_tf  = vec2.transform(X2_test)

clf2 = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
clf2.fit(X2_train_tf, y2_train)

print("\n=== Task 2: Severity Tier (in-distribution 80/20) ===")
print(classification_report(y2_test, clf2.predict(X2_test_tf)))

Task 2 corpus: 1368 rows
severity_tier
Medium    840
High      402
Low       126
Name: count, dtype: int64

=== Task 2: Severity Tier (in-distribution 80/20) ===
              precision    recall  f1-score   support

        High       0.68      0.67      0.67        81
         Low       0.17      0.08      0.11        25
      Medium       0.75      0.82      0.78       168

    accuracy                           0.70       274
   macro avg       0.53      0.52      0.52       274
weighted avg       0.68      0.70      0.69       274



In [7]:
eu2 = task2[task2['jurisdiction'] != 'US'].copy()
us2 = task2[task2['jurisdiction'] == 'US'].copy()

print(f"EU: {len(eu2)} rows | US: {len(us2)} rows")

# Train EU → Test US
vec_eu2 = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_eu2_tf = vec_eu2.fit_transform(eu2['decision_text'].astype(str))
clf_eu2 = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
clf_eu2.fit(X_eu2_tf, eu2['severity_tier'])

X_us2_tf = vec_eu2.transform(us2['decision_text'].astype(str))
print("\n=== Task 2: Train EU → Test US ===")
print(classification_report(us2['severity_tier'], clf_eu2.predict(X_us2_tf)))

# Train US → Test EU
vec_us2 = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_us2_tf2 = vec_us2.fit_transform(us2['decision_text'].astype(str))
clf_us2 = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
clf_us2.fit(X_us2_tf2, us2['severity_tier'])

X_eu2_tf2 = vec_us2.transform(eu2['decision_text'].astype(str))
print("\n=== Task 2: Train US → Test EU ===")
print(classification_report(eu2['severity_tier'], clf_us2.predict(X_eu2_tf2)))

EU: 1222 rows | US: 146 rows

=== Task 2: Train EU → Test US ===
              precision    recall  f1-score   support

        High       0.80      0.94      0.86       113
      Medium       0.46      0.18      0.26        33

    accuracy                           0.77       146
   macro avg       0.63      0.56      0.56       146
weighted avg       0.72      0.77      0.73       146


=== Task 2: Train US → Test EU ===
              precision    recall  f1-score   support

        High       0.24      1.00      0.38       289
         Low       0.00      0.00      0.00       126
      Medium       0.00      0.00      0.00       807

    accuracy                           0.24      1222
   macro avg       0.08      0.33      0.13      1222
weighted avg       0.06      0.24      0.09      1222



In [10]:
# Rebuild on reset index for clean alignment
task2_r = task2.reset_index(drop=True)
task2_r['year'] = pd.to_datetime(task2_r['decision_date'], errors='coerce').dt.year.fillna(2020).astype(int)

tr_idx, te_idx, y_tr, y_te = train_test_split(
    task2_r.index.tolist(), task2_r['severity_tier'].tolist(),
    test_size=0.2, random_state=42, stratify=task2_r['severity_tier']
)

# Metadata: jurisdiction (one-hot) + year (numeric)
enc = OneHotEncoder(handle_unknown='ignore')
jur_tr = enc.fit_transform(task2_r.loc[tr_idx, ['jurisdiction']])
jur_te = enc.transform(task2_r.loc[te_idx, ['jurisdiction']])

year_tr = csr_matrix(task2_r.loc[tr_idx, ['year']].values)
year_te = csr_matrix(task2_r.loc[te_idx, ['year']].values)

X_meta_tr = hstack([jur_tr, year_tr])
X_meta_te = hstack([jur_te, year_te])

# Text features
vec3 = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_text_tr = vec3.fit_transform(task2_r.loc[tr_idx, 'decision_text'].astype(str))
X_text_te = vec3.transform(task2_r.loc[te_idx, 'decision_text'].astype(str))

# Metadata-only
clf_m = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=777)
clf_m.fit(X_meta_tr, y_tr)
print("=== Ruohonen: Metadata-only (jurisdiction + year) ===")
print(classification_report(y_te, clf_m.predict(X_meta_te)))

# Text-only
clf_t = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=777)
clf_t.fit(X_text_tr, y_tr)
print("=== Ruohonen: Text-only ===")
print(classification_report(y_te, clf_t.predict(X_text_te)))

# Combined
clf_c = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=777)
clf_c.fit(hstack([X_text_tr, X_meta_tr]), y_tr)
print("=== Ruohonen: Text + Metadata ===")
print(classification_report(y_te, clf_c.predict(hstack([X_text_te, X_meta_te]))))

=== Ruohonen: Metadata-only (jurisdiction + year) ===
              precision    recall  f1-score   support

        High       0.65      0.60      0.63        81
         Low       0.13      0.48      0.20        25
      Medium       0.71      0.44      0.54       168

    accuracy                           0.49       274
   macro avg       0.50      0.51      0.46       274
weighted avg       0.64      0.49      0.54       274

=== Ruohonen: Text-only ===
              precision    recall  f1-score   support

        High       0.71      0.73      0.72        81
         Low       0.18      0.12      0.14        25
      Medium       0.76      0.79      0.78       168

    accuracy                           0.71       274
   macro avg       0.55      0.55      0.55       274
weighted avg       0.69      0.71      0.70       274

=== Ruohonen: Text + Metadata ===
              precision    recall  f1-score   support

        High       0.70      0.74      0.72        81
         Low 

In [ ]:
## Results Summary

### Task 1: Violation Type (coarse_label)
| Setting | Macro F1 |
|---|---|
| In-distribution (80/20) | 0.75 |
| Train EU → Test US | 0.23 |
| Train US → Test EU | 0.05 |

TF-IDF achieves strong in-distribution performance. Cross-jurisdiction transfer collapses, confirming that surface vocabulary does not transfer across legal regimes. Motivates Legal-BERT as a semantic alternative.

### Task 2: Severity Tier (Low / Medium / High)
| Setting | Macro F1 |
|---|---|
| In-distribution (80/20) | 0.55 |
| Train EU → Test US | 0.56 |
| Train US → Test EU | 0.13 |

### Ruohonen Challenge
| Model | Macro F1 |
|---|---|
| Metadata-only (jurisdiction + year) | 0.46 |
| Text-only (TF-IDF) | 0.55 |
| Text + Metadata | 0.56 |

Text outperforms metadata for severity prediction, partially challenging Ruohonen & Hjerppe (2020). Caveat: their metadata was richer (sector, controller type, articles). Low tier is the exception — jurisdiction predicts Low better than text, consistent with Low being an EU-dominant phenomenon. Combined model adds marginal improvement, suggesting metadata adds little beyond what text already encodes.